# AstroLogic — RL Training & Comparison

Train **3 RL algorithms** (DQN, PPO, REINFORCE) with **10 hyperparameter configurations each** (30 total experiments) against the same `AstroExploration-v0` environment.

**High-reward exploitation strategies:**
| Algorithm | Mechanism | How It Works |
|-----------|-----------|-------------|
| DQN | Prioritized Experience Replay (PER) | High TD-error transitions sampled more often — rare positive outcomes exploited |
| PPO | Advantage normalization + high `n_epochs` | Above-average returns → positive advantage → policy strengthened; below-average → suppressed |
| REINFORCE | Baseline subtraction (mean / running) | Returns above baseline → positive gradient → policy reinforced; below → moved away |

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Resolve the project root — search common locations for the environment package.
def _has_env_package(p):
    return (p / "environment").is_dir() or (p / "astro_env").is_dir()

project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path(r"c:\Users\LENOVO\Desktop\summative_rl\AstroLogic_beta"),
    Path("/kaggle/working"),
    Path("/mnt/data"),
    Path("/mnt/workspace"),
    Path("/workspace"),
    Path("/workspaces"),
    Path("/home"),
]

# Also search one level deep under /kaggle/input (dataset subdirs)
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    project_candidates += list(kaggle_input.iterdir())

project_root = next((c for c in project_candidates if _has_env_package(c)), None)

if project_root is None:
    project_root = Path.cwd()

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Using project root: {project_root}")

try:
    import environment  # registers AstroExploration-v0
except ModuleNotFoundError:
    import astro_env  # registers AstroExploration-v0
import gymnasium as gym

from training.dqn_training import train_dqn, DQN_CONFIGS
from training.pg_training import train_ppo, PPO_CONFIGS, train_reinforce, REINFORCE_CONFIGS

print(f"DQN configs:       {len(DQN_CONFIGS)}")
print(f"PPO configs:       {len(PPO_CONFIGS)}")
print(f"REINFORCE configs: {len(REINFORCE_CONFIGS)}")
print(f"Total experiments: {len(DQN_CONFIGS) + len(PPO_CONFIGS) + len(REINFORCE_CONFIGS)}")

Using project root: /kaggle/working


ModuleNotFoundError: No module named 'astro_env'

In [2]:
# Verify environment spaces
env = gym.make("AstroExploration-v0")
print("Observation space:", env.observation_space)
print("Observation shape:", env.observation_space.shape)
print("Action space:     ", env.action_space)
print("Action nvec:      ", env.action_space.nvec)
print()

obs, info = env.reset()
print("Sample observation (shape={}):\n{}".format(obs.shape, obs))
print("\nInfo keys:", list(info.keys()))
env.close()

NameError: name 'gym' is not defined

## DQN Training (10 runs)

All configs use **Prioritized Experience Replay (PER)** via `sb3-contrib`. PER samples transitions with high TD-error more often, which correlates with surprising / high-reward transitions. This lets the agent learn more from rare positive outcomes and forget ineffective transitions.

In [ ]:
dqn_results = []
for i in range(len(DQN_CONFIGS)):
    result = train_dqn(i)
    dqn_results.append(result)
print(f"\nAll {len(dqn_results)} DQN runs complete.")

In [ ]:
dqn_df = pd.DataFrame(dqn_results)
dqn_df

## PPO Training (10 runs)

PPO exploits high-reward rollouts through **advantage normalization**: above-average returns get positive advantages (policy strengthened), below-average returns get negative advantages (policy suppressed). Higher `n_epochs` means more exploitation per rollout buffer.

In [ ]:
ppo_results = []
for i in range(len(PPO_CONFIGS)):
    result = train_ppo(i)
    ppo_results.append(result)
print(f"\nAll {len(ppo_results)} PPO runs complete.")

In [ ]:
ppo_df = pd.DataFrame(ppo_results)
ppo_df

## REINFORCE Training (10 runs)

REINFORCE (Monte Carlo Policy Gradient) exploits high-reward episodes via **baseline subtraction**: returns above the baseline produce positive policy gradients (reinforcing good actions), returns below the baseline produce negative gradients (discouraging poor actions). Three baseline modes are tested: `none`, `mean`, and `running`.

In [ ]:
reinforce_results = []
for i in range(len(REINFORCE_CONFIGS)):
    result = train_reinforce(i)
    reinforce_results.append(result)
print(f"\nAll {len(reinforce_results)} REINFORCE runs complete.")

In [ ]:
reinforce_df = pd.DataFrame(reinforce_results)
reinforce_df

## Cross-Algorithm Comparison

Load training logs from all 30 runs and plot learning curves, best-run bar chart, and a summary table.

In [ ]:
import glob

def load_monitor_rewards(log_dir):
    """Load episode rewards from SB3 Monitor CSV (skips first comment line)."""
    csv_path = os.path.join(log_dir, "monitor.csv")
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path, skiprows=1)
    if "r" in df.columns:
        return df["r"].values
    return None

def load_reinforce_rewards(model_dir):
    """Load episode rewards from REINFORCE rewards.csv."""
    csv_path = os.path.join(model_dir, "rewards.csv")
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path)
    if "reward" in df.columns:
        return df["reward"].values
    return None

# Collect all reward curves
algo_curves = {"DQN": [], "PPO": [], "REINFORCE": []}

# DQN runs
for cfg in DQN_CONFIGS:
    log_dir = f"models/dqn/{cfg['name']}/logs"
    rewards = load_monitor_rewards(log_dir)
    if rewards is not None:
        algo_curves["DQN"].append((cfg["name"], rewards))

# PPO runs
for cfg in PPO_CONFIGS:
    log_dir = f"models/pg/{cfg['name']}/logs"
    rewards = load_monitor_rewards(log_dir)
    if rewards is not None:
        algo_curves["PPO"].append((cfg["name"], rewards))

# REINFORCE runs
for cfg in REINFORCE_CONFIGS:
    model_dir = f"models/pg/{cfg['name']}"
    rewards = load_reinforce_rewards(model_dir)
    if rewards is not None:
        algo_curves["REINFORCE"].append((cfg["name"], rewards))

for algo, curves in algo_curves.items():
    print(f"{algo}: {len(curves)} runs loaded")

In [ ]:
# Learning curves — 1x3 subplots with rolling average
COLORS = {"DQN": "#e74c3c", "PPO": "#2ecc71", "REINFORCE": "#9b59b6"}
WINDOW = 50

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, algo in zip(axes, ["DQN", "PPO", "REINFORCE"]):
    curves = algo_curves[algo]
    if not curves:
        ax.set_title(f"{algo} (no data)")
        continue
    for name, rewards in curves:
        rolling = pd.Series(rewards).rolling(WINDOW, min_periods=1).mean()
        ax.plot(rolling, alpha=0.6, linewidth=0.8, label=name)
    ax.set_title(f"{algo} Learning Curves", fontsize=13)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward (rolling avg)")
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(True, alpha=0.3)

fig.suptitle("AstroLogic — Training Learning Curves (30 runs)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: best run per algorithm (mean reward of final 50 episodes)
best_runs = {}
for algo, curves in algo_curves.items():
    if not curves:
        continue
    best_name, best_mean = None, -float("inf")
    for name, rewards in curves:
        tail_mean = np.mean(rewards[-50:]) if len(rewards) >= 50 else np.mean(rewards)
        if tail_mean > best_mean:
            best_mean = tail_mean
            best_name = name
    best_runs[algo] = (best_name, best_mean)

algos = list(best_runs.keys())
means = [best_runs[a][1] for a in algos]
names = [best_runs[a][0] for a in algos]
colors = [COLORS[a] for a in algos]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(algos, means, color=colors, edgecolor="black", linewidth=0.8)
for bar, name, mean in zip(bars, names, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{name}\n({mean:.1f})", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Mean Reward (final 50 episodes)")
ax.set_title("Best Run per Algorithm", fontsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table: all 30 runs
rows = []
for algo, curves in algo_curves.items():
    for name, rewards in curves:
        tail_mean = np.mean(rewards[-50:]) if len(rewards) >= 50 else np.mean(rewards)
        rows.append({
            "algorithm": algo,
            "run_name": name,
            "total_episodes": len(rewards),
            "final_mean_reward": round(tail_mean, 2),
        })

# Add wall times from training results
wall_times = {}
for r in dqn_results + ppo_results + reinforce_results:
    wall_times[r["run_name"]] = r["wall_time"]

summary_df = pd.DataFrame(rows)
summary_df["wall_time_s"] = summary_df["run_name"].map(wall_times).round(1)
summary_df = summary_df.sort_values("final_mean_reward", ascending=False).reset_index(drop=True)
summary_df